# Linear Regression Refresh: Auto MPG
## Class Example — Medina County Career Center

**Goal:** Predict a car's miles-per-gallon (MPG) from its specs.

**Dataset:** Auto MPG from UCI Machine Learning Repository (398 cars, 7 features)

**The process:**
1. Load the data
2. Explore + check correlations
3. Train/test split
4. Build the model
5. Evaluate (R², MAE)
6. Interpret coefficients
7. Visualize

## Step 0: Install + Import Libraries

Run this once to install the UCI dataset loader, then import everything.

In [ ]:
# Run this cell ONCE to install the UCI dataset package
!pip install ucimlrepo -q

In [ ]:
# Import all libraries
import pandas as pd                                        # data tables
import numpy as np                                         # math
import matplotlib.pyplot as plt                            # charts
import seaborn as sns                                      # prettier charts
from sklearn.model_selection import train_test_split       # split data 80/20
from sklearn.linear_model import LinearRegression          # the model
from sklearn.metrics import r2_score, mean_absolute_error  # evaluation
from ucimlrepo import fetch_ucirepo                        # UCI dataset loader

print('All libraries loaded!')

## Step 1: Load the Data

The Auto MPG dataset has 398 cars from the late 1970s-80s. Each row is a car with:
- **Target (y):** mpg — miles per gallon
- **Features (X):** cylinders, displacement, horsepower, weight, acceleration, model_year, origin

In [ ]:
# Fetch the Auto MPG dataset from UCI (id=9)
autoMpg = fetch_ucirepo(id=9)

# Separate features (X) and target (y)
X = autoMpg.data.features    # the input columns
y = autoMpg.data.targets     # what we're predicting (mpg)

# Combine into one DataFrame for exploration
df = pd.concat([X, y], axis=1)

print(f'Dataset shape: {df.shape[0]} cars, {df.shape[1]} columns')
print(f'\nFirst 5 rows:')
df.head()

## Step 2: Clean the Data

Check for missing values and handle them. Real data is messy!

In [ ]:
# Check for missing values
print('Missing values per column:')
print(df.isnull().sum())
print(f'\nTotal rows before cleaning: {len(df)}')

# Drop rows with any missing values
df = df.dropna()
print(f'Total rows after cleaning: {len(df)}')

# Update X and y after cleaning
X = df.drop('mpg', axis=1)
y = df['mpg']

## Step 3: Explore — Correlations

Which features are most correlated with MPG? Remember:
- **r close to +1 or -1** = strong relationship
- **r close to 0** = weak/no relationship

In [ ]:
# Calculate Pearson r for every pair of columns
correlationMatrix = df.corr()

# Show just the correlations with MPG, sorted by strength
mpgCorrelations = correlationMatrix['mpg'].drop('mpg').sort_values()
print('Correlations with MPG (sorted):\n')
for feature, r in mpgCorrelations.items():
    r2 = r ** 2
    print(f'  {feature:15s}  r = {r:+.3f}   R² = {r2:.3f} ({r2*100:.0f}%)')

In [ ]:
# Correlation heatmap
plt.figure(figsize=(9, 7))
sns.heatmap(correlationMatrix, annot=True, cmap='coolwarm', center=0,
            fmt='.2f', vmin=-1, vmax=1)
plt.title('Correlation Heatmap — Auto MPG', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Red = positive correlation, Blue = negative correlation')
print('Notice: weight has the strongest negative correlation with MPG')

In [ ]:
# Scatter plot: weight vs MPG (strongest correlation)
plt.figure(figsize=(8, 5))
plt.scatter(df['weight'], df['mpg'], alpha=0.5, color='steelblue', s=30)
plt.xlabel('Weight (lbs)', fontsize=11)
plt.ylabel('Miles Per Gallon', fontsize=11)
plt.title('Car Weight vs MPG', fontsize=13, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

r = df['weight'].corr(df['mpg'])
print(f'Pearson r = {r:.3f}')
print(f'R² = {r**2:.3f} — weight alone explains {r**2*100:.0f}% of MPG variation')

## Step 4: Train/Test Split

Split the data: 80% to train the model, 20% to test it on data it's never seen.

In [ ]:
# Split: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training set: {len(X_train)} cars')
print(f'Test set:     {len(X_test)} cars (model has NEVER seen these)')

## Step 5: Build the Model

Create a Linear Regression model and train it. The model will find the best coefficients.

In [ ]:
# Create and train the model
model = LinearRegression()
model.fit(X_train, y_train)

print('Model trained!')
print(f'Found {len(model.coef_)} coefficients + 1 intercept')

## Step 6: Evaluate the Model

How good is our model? Check R² and MAE on the TEST data (data it never saw during training).

In [ ]:
# Make predictions on the test set
predictions = model.predict(X_test)

# Calculate metrics
r2 = r2_score(y_test, predictions)
mae = mean_absolute_error(y_test, predictions)

print('MODEL PERFORMANCE (on test data):')
print(f'  R² Score: {r2:.4f} ({r2*100:.1f}%)')
print(f'  MAE:      {mae:.2f} MPG')
print(f'\nInterpretation:')
print(f'  The model explains {r2*100:.1f}% of the variation in MPG.')
print(f'  On average, predictions are off by about {mae:.1f} MPG.')

## Step 7: Interpret the Coefficients

What did the model learn? Each coefficient tells us how much that feature affects MPG.

In [ ]:
# Display the coefficients
coeffDf = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_
}).sort_values('Coefficient', key=abs, ascending=False)

print('WHAT THE MODEL LEARNED:\n')
for _, row in coeffDf.iterrows():
    direction = 'increases' if row['Coefficient'] > 0 else 'decreases'
    print(f"  {row['Feature']:15s}  coeff = {row['Coefficient']:+.4f}")

print(f"\n  Intercept: {model.intercept_:.2f}")
print(f'\nPositive coefficient = pushes MPG UP')
print(f'Negative coefficient = pushes MPG DOWN')

In [ ]:
# Bar chart of coefficients
plt.figure(figsize=(8, 4))
colors = ['green' if c > 0 else 'red' for c in coeffDf['Coefficient']]
plt.barh(coeffDf['Feature'], coeffDf['Coefficient'], color=colors)
plt.xlabel('Effect on MPG', fontsize=11)
plt.title('Which Features Affect MPG Most?', fontsize=13, fontweight='bold')
plt.axvline(x=0, color='black', linewidth=0.8)
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## Step 8: Visualize Predictions vs Actual

In [ ]:
# Actual vs Predicted scatter plot
plt.figure(figsize=(8, 6))
plt.scatter(y_test, predictions, alpha=0.5, color='steelblue', s=40,
            label='Our predictions')
# Perfect prediction line
minVal = min(y_test.min(), predictions.min())
maxVal = max(y_test.max(), predictions.max())
plt.plot([minVal, maxVal], [minVal, maxVal], 'r--', linewidth=2,
         label='Perfect predictions')

plt.xlabel('Actual MPG', fontsize=11)
plt.ylabel('Predicted MPG', fontsize=11)
plt.title('How Close Are Our Predictions?', fontsize=13, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('Dots close to the red line = good predictions')
print('Dots far from the red line = the model was off')

## Step 9: Make a Prediction

Use the model to predict MPG for a hypothetical car. Try changing the values!

In [ ]:
# Create a hypothetical car
myCar = pd.DataFrame({
    'cylinders': [4],
    'displacement': [120],
    'horsepower': [80],
    'weight': [2500],
    'acceleration': [16],
    'model_year': [82],
    'origin': [1]
})

predictedMpg = model.predict(myCar)[0]
print(f'Hypothetical car specs:')
print(f'  4 cylinders, 120 displacement, 80 HP')
print(f'  2500 lbs, 16 sec acceleration, 1982, US-made')
print(f'\nPredicted MPG: {predictedMpg:.1f}')

---

## Summary

**What we did:**
1. Loaded real car data from UCI (398 cars)
2. Explored correlations — weight has the strongest relationship with MPG
3. Split data 80/20 for honest evaluation
4. Built a Linear Regression model
5. Evaluated with R² and MAE
6. Interpreted coefficients to understand what the model learned
7. Visualized predictions vs actual values

**Key takeaways:**
- The process is always the same: load → clean → explore → split → build → evaluate
- Coefficients tell you HOW the model makes decisions
- R² tells you how GOOD the model is
- This same process works for ANY regression problem — just change the dataset!